# Data loading

In [ ]:
import json
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
import re
from typing import Tuple


The trial release includes: **one ALD/E research paper**, and its associated **annotated scientific figures**, with a focus on quantitative plots.

### 🗂️ Directory Structure

```text
icdar2026-competition-data
├── trial
│   ├── main-category
│   │   ├── sub-category
│   │   │   ├── paper #
│   │   │   │   ├── images
│   │   │   │   │   ├── filename.jpg            # (JPEG) actual figure image
│   │   │   │   │   ├── filename.json           # (JSON) figure annotations
│   │   │   │   │   └── ...
│   │   │   │   ├── Author et al.pdf                # (PDF) actual PDF document
│   │   │   │   ├── content.json                    # (JSON) structured content
│   │   │   │   └── ...
│   │   │   └── ...
```

In [ ]:
!git clone https://github.com/sciknoworg/ALD-E-ImageMiner/ ../ALD-E-ImageMiner

In [ ]:
MODEL_ID = "unsloth/Qwen3.5-0.8B"

BASE_DIR = Path.cwd().parent
CATEGORY = "train"

COMPETITION_DATA_DIR = BASE_DIR / "ALD-E-ImageMiner" / "icdar2026-competition-data"
CASE_DIR = COMPETITION_DATA_DIR / CATEGORY

assert CASE_DIR.is_dir(), "Dataset not found"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


### 📝 Annotations

Following is the general top-level schema of each JSON file. Each "dict" contains alphabetical keys-values pairs to represent each sub-figure task-specific data as shown below, where

- **sample_id**: Unique sample id, represents actual path to the figure
- **classification**: Data for the classification task 
- **summarization**: Data for the Summarization task
- **data_extraction**: Data for the Data extraction task
- **vqa**: Data for the VQA task
- **bbox**: Bounding box coordinates to extract each sub-figure

```json
{
    "sample_id": str,
    "classification": {
        "a": str,
        "b": str,
        ...
    },
    "summarization": {
        "a": str,
        "b": str,
        ...
    },
    "data_extraction": {
        "a": str,
        "b": str,
        ...
    },
    "vqa": {
        "a": {
            "question_type": str,
            "questions": str,
            "answer_type": str,
            "answer": str
        },
        "b": list,
        ...
    },
    "bbox": {
        "a": {
            "x": int,
            "y": int,
            "width": int,
            "height": int
        },
        "b": {
            "x": int,
            "y": int,
            "width": int,
            "height": int
        },
        ...
    },
}
```

In [ ]:
!ls -R {CASE_DIR}

In [ ]:
!cat {CASE_DIR}/atomic-layer-etching/experimental-usecase/16/images/fig1.json

In [ ]:
def analyze_dataset(case_dir: Path, tokenizer, max_samples=None):
    stats = []
    json_files = list(case_dir.rglob("*.json"))

    if max_samples:
        json_files = json_files[:max_samples]

    pbar = tqdm(json_files, desc="Analyzing Data")

    for json_file in pbar:
        fullpath = str(json_file)
        if "images" not in fullpath or ".vscode" in fullpath:
            continue

        pbar.set_description(f"Processing {json_file.name}")

        with open(json_file, "r") as f:
            data = json.load(f)

        bboxes = data.get("bbox", {})

        for sub_key, q_list in data.get("vqa", {}).items():
            if sub_key not in bboxes:
                continue

            # Get coordinates to calculate crop size
            box = bboxes[sub_key]
            width = box["width"]
            height = box["height"]

            for q_obj in q_list:
                gt_response = q_obj.get("answer", "")

                answer_tokens = len(
                    tokenizer.encode(gt_response, add_special_tokens=False)
                )

                stats.append(
                    {
                        "file": json_file.name,
                        "subfigure": sub_key,
                        "answer_tokens": answer_tokens,
                        "image_width": width,
                        "image_height": height,
                        "image_pixels": width * height,
                    }
                )

    return pd.DataFrame(stats)

In [ ]:
df_stats = analyze_dataset(CASE_DIR, tokenizer)

### 📊 Text Token Statistics

Let's look at the distribution of text tokens. This tells us how much context length is consumed purely by the text prompt and the desired output.

> 1. To set `MAX_NEW_TOKENS`: Look at the 99th percentile of `answer_tokens`.
> 2. To set `max_length` in `SFTConfig`: Add the 99th percentile of `answer_tokens` + `prompt_tokens` + `visual_tokens` to the estimated visual tokens. 
> 
> *Qwen typically encodes images dynamically. If your average image is ~1000x1000, it can easily take up 1000+ tokens on its own. A safe `max_length` is usually between 2048 and 4096 depending on your hardware limits.*

In [ ]:
display(df_stats["answer_tokens"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

In [ ]:
df_stats["answer_tokens"].hist(bins=50, color="skyblue", edgecolor="black")
plt.show()

### 🖼️ Image Dimension Statistics

Qwen-VL processes images by splitting them into patches. Larger images consume exponentially more sequence length. If your subfigure crops are massive, you may need a much higher `max_length` or you might need to resize them before training.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].scatter(
    df_stats["image_width"], df_stats["image_height"], alpha=0.5, color="green"
)
axes[0].set_title("Image Crop Dimensions (Width vs Height)")
axes[0].set_xlabel("Width (px)")
axes[0].set_ylabel("Height (px)")

df_stats["image_pixels"].hist(bins=50, ax=axes[1], color="purple", edgecolor="black")
axes[1].set_title("Distribution of Image Area (Pixels)")
axes[1].set_xlabel("Total Pixels")
axes[1].set_ylabel("Frequency")
plt.show()

In [ ]:
def clean_and_validate_output(raw_answer: str, expected_type: str) -> Tuple[str, bool]:
    """
    Cleans the answer and checks if it conforms to the expected type.
    Returns: (cleaned_answer, is_valid_format)
    """
    if not raw_answer or not isinstance(raw_answer, str):
        return "", False

    # Basic cleaning
    cleaned = raw_answer.strip()

    if expected_type == "Yes/No":
        ans_lower = cleaned.lower()
        # Use regex word boundaries (\b) to avoid matching "eyes", "note", "nothing", etc.
        if re.search(r"\byes\b", ans_lower):
            return "Yes", True
        elif re.search(r"\bno\b", ans_lower):
            return "No", True
        return cleaned, False

    elif expected_type == "List":
        # Split by comma, strip whitespace, and drop any empty strings
        elements = [item.strip() for item in cleaned.split(",") if item.strip()]
        return ", ".join(elements), len(elements) > 0

    elif expected_type == "Factoid":
        # Remove trailing punctuation often mistakenly added by LLMs to short terms
        cleaned_factoid = cleaned.strip()
        return cleaned_factoid, len(cleaned_factoid) > 0

    elif expected_type == "Paragraph":
        # Replace multiple spaces/newlines with single spaces for a clean paragraph
        cleaned_para = re.sub(r"\s+", " ", cleaned).strip()
        sentences = [s for s in re.split(r"[.!?]+", cleaned_para) if s.strip()]
        return cleaned_para, len(sentences) >= 1

    return cleaned, True


def report_and_fix_questions(case_dir: Path, target_answer_type: str = "Yes/No"):
    """
    Iterates through the dataset, attempts to clean target answers, categorizes them,
    prints a summary, and persists the full state to the filesystem.
    """
    json_files = list(case_dir.rglob("*.json"))

    report_data = {
        "summary": {"total_processed": 0, "perfect": 0, "fixed": 0, "unfixable": 0},
        "details": {
            "perfect": [],  # Required no changes
            "fixed": [],  # Required changes but successfully validated
            "unfixable": [],  # Could not be formatted correctly
        },
    }

    print(f"--- Analyzing '{target_answer_type}' questions ---\n")

    for json_file in json_files:
        if "images" not in str(json_file) or ".vscode" in str(json_file):
            continue

        with open(json_file, "r") as f:
            data = json.load(f)

        for sub_key, q_list in data.get("vqa", {}).items():
            for q_obj in q_list:
                answer_type = q_obj.get("answer_type", "")
                if answer_type != target_answer_type:
                    continue

                report_data["summary"]["total_processed"] += 1

                actual_answer = q_obj.get("answer", "")
                question_text = q_obj.get("question") or q_obj.get("questions", "")

                # Attempt to clean and validate
                cleaned_answer, is_valid = clean_and_validate_output(
                    actual_answer, target_answer_type
                )

                record = {
                    "file": json_file.name,
                    "subfigure": sub_key,
                    "question": question_text,
                    "actual_answer": actual_answer,
                    "cleaned_answer": cleaned_answer,
                }

                # Categorize the result
                if not is_valid:
                    report_data["summary"]["unfixable"] += 1
                    report_data["details"]["unfixable"].append(record)
                    # print(f"[UNFIXABLE] File: {json_file.name} | Sub: {sub_key}")
                    print(f"  Question: {question_text}")
                    print(f"  Actual: {actual_answer}")
                    print(f"  Cleaned: {cleaned_answer}\n")

                elif actual_answer == cleaned_answer:
                    report_data["summary"]["perfect"] += 1
                    report_data["details"]["perfect"].append(record)

                else:
                    report_data["summary"]["fixed"] += 1
                    report_data["details"]["fixed"].append(record)
                    # print(f"[FIXED] File: {json_file.name} | Sub: {sub_key}")
                    print(f"  Question: {question_text}")
                    print(f"  Actual: {actual_answer}")
                    print(f"  Cleaned:  {cleaned_answer}\n")

    print("-" * 40)
    print("📋 SUMMARY REPORT")
    print(f"Total Processed: {report_data['summary']['total_processed']}")
    print(f"Perfect (No fix needed): {report_data['summary']['perfect']}")
    print(f"Fixed (Cleaned & Validated): {report_data['summary']['fixed']}")
    print(f"Unfixable (Invalid format): {report_data['summary']['unfixable']}")

In [ ]:
report_and_fix_questions(CASE_DIR, target_answer_type="Factoid")

In [ ]:
def report_yes_no_balance(case_dir: Path):
    """
    Iterates through the dataset, attempts to clean target answers, categorizes them,
    prints a summary, and persists the full state to the filesystem.
    """
    json_files = list(case_dir.rglob("*.json"))

    report_data = {
        "summary": {"total_processed": 0, "yes": 0, "no": 0},
        "details": {
            "yes": [],  # Answer is "Yes"
            "no": [],  # Answer is "No"
        },
    }

    target_answer_type: str = "Yes/No"

    for json_file in json_files:
        if "images" not in str(json_file) or ".vscode" in str(json_file):
            continue

        with open(json_file, "r") as f:
            data = json.load(f)

        for sub_key, q_list in data.get("vqa", {}).items():
            for q_obj in q_list:
                answer_type = q_obj.get("answer_type", "")
                if answer_type != target_answer_type:
                    continue

                report_data["summary"]["total_processed"] += 1

                actual_answer = q_obj.get("answer", "")

                # Attempt to clean and validate
                cleaned_answer, is_valid = clean_and_validate_output(
                    actual_answer, target_answer_type
                )

                if cleaned_answer == "Yes":
                    report_data["summary"]["yes"] += 1
                elif cleaned_answer == "No":
                    report_data["summary"]["no"] += 1

    print("-" * 40)
    print("✅ YES/NO BALANCE REPORT")
    print(f"Total Processed: {report_data['summary']['total_processed']}")
    print(f"Yes: {report_data['summary']['yes']}")
    print(f"No: {report_data['summary']['no']}")

In [ ]:
report_yes_no_balance(CASE_DIR)

In [ ]:
def report_missing_data_extraction(case_dir: Path):
    """
    Calculates the count and percentage of VQA questions associated with
    subfigures that lack a 'data_extraction' entry in the annotations.
    """
    json_files = list(case_dir.rglob("*.json"))

    total_questions = 0
    missing_extraction_questions = 0

    for json_file in json_files:
        if "images" not in str(json_file) or ".vscode" in str(json_file):
            continue

        with open(json_file, "r", encoding="utf-8") as f:
            data = json.load(f)

        vqa_data = data.get("vqa", {})
        data_ext = data.get("data_extraction", {})

        for sub_key, q_list in vqa_data.items():
            num_questions = len(q_list)
            total_questions += num_questions

            # Check if this subfigure is missing from the data_extraction dict
            if sub_key not in data_ext:
                missing_extraction_questions += num_questions

    if total_questions == 0:
        print("No VQA questions found in the specified directory.")
        return 0, 0, 0.0

    percentage = (missing_extraction_questions / total_questions) * 100

    print("-" * 40)
    print("📊 DATA EXTRACTION COMPLETENESS REPORT")
    print("-" * 40)
    print(f"Total VQA Questions: {total_questions}")
    print(f"Questions missing data extraction: {missing_extraction_questions}")
    print(f"Percentage missing: {percentage:.2f}%")
    print("-" * 40)

    return total_questions, missing_extraction_questions, percentage

In [ ]:
_ = report_missing_data_extraction(CASE_DIR)